In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# create folder to keep figures
# figure output directory
output_dir = r"./../../dataFolders/MuscaChasingBeads/Figures/Interpolation/"
os.makedirs(output_dir, exist_ok=True)

In [3]:
# Where interpolated CSVs live
interpol_dir = r"./../../dataFolders/MuscaChasingBeads/xyz_Interpolated/"

# Base directory containing xyz points Trial2_200rpm, Trial3_180rpm, etc.
raw_dir = r"./../../dataFolders/MuscaChasingBeads/xyz_Raw_Trimmed/"


# ==========================================================
# POINT DEFINITIONS
# ==========================================================

points = {
    "pt1": "Bead",
    "pt2": "Head",
    "pt3": "Left Wing Base",
    "pt4": "Right Wing Base",
    "pt5": "Left Wing Tip",
    "pt6": "Right Wing Tip",
    "pt7": "Abdomen Tip"
}

# ==========================================================
# FIND INTERPOLATED FILES
# ==========================================================

interp_csv_files = sorted(glob.glob(os.path.join(interpol_dir, "*_INTERP.csv")))
if not interp_csv_files:
    raise FileNotFoundError("No *_INTERP.csv files found")

raw_csv_files = sorted(glob.glob(os.path.join(raw_dir, "*_trimmed.csv")))
if not raw_csv_files:
    raise FileNotFoundError("No *_raw.csv files found")

In [4]:
# ==========================================================
# PLOTTING
# ==========================================================

for path in interp_csv_files:
    fname = os.path.splitext(os.path.basename(path))[0]
    print("Plotting:", fname)

    # ------------------------------------------------------
    # Extract trial name (e.g. Trial2_200rpm)
    # ------------------------------------------------------
    trial_name = '_'.join(fname.split("_")[0:2])[:-6]
#     try:
        
#     except IndexError:
#         raise ValueError(f"Could not extract trial name from: {fname}")

    # ------------------------------------------------------
    # Create trial-specific INTERP_PLOTS folder
    # ------------------------------------------------------
    fig_dir = os.path.join(output_dir, trial_name)
    os.makedirs(fig_dir, exist_ok=True)

    # ------------------------------------------------------
    # Load data
    # ------------------------------------------------------
    df = pd.read_csv(path)
    frames = df["frame"].values
    
    raw_df = pd.read_csv([p for p in raw_csv_files if trial_name in p][0])

    # ------------------------------------------------------
    # Collect interpolated XYZ data
    # ------------------------------------------------------
    interp = {}
    for pt in points.keys():
        interp[pt] = np.column_stack([
            df[f"{pt}_X"].values,
            df[f"{pt}_Y"].values,
            df[f"{pt}_Z"].values,
        ])
        
    raw = {}
    for pt in points.keys():
        raw[pt] = np.column_stack([
            raw_df[f"{pt}_X"].values,
            raw_df[f"{pt}_Y"].values,
            raw_df[f"{pt}_Z"].values,
        ])

    # ------------------------------------------------------
    # CENTER (robust handling)
    # ------------------------------------------------------
    if all(c in df.columns for c in ["center_X", "center_Y", "center_Z"]):
        # Use center already present in CSV
        interp["center"] = np.column_stack([
            df["center_X"].values,
            df["center_Y"].values,
            df["center_Z"].values,
        ])
        
        raw["center"] = np.column_stack([
            raw_df["center_X"].values,
            raw_df["center_Y"].values,
            raw_df["center_Z"].values,
        ])
    else:
        # Compute center from wing bases
        interp["center"] = 0.5 * (interp["pt3"] + interp["pt4"])
        raw["center"] = 0.5 * (raw["pt3"] + raw["pt4"])

    points_with_center = dict(points)
    points_with_center["center"] = "Center"

    # ------------------------------------------------------
    # Plot XYZ vs frame
    # ------------------------------------------------------
    for pt, label in points_with_center.items():
        #         for i, axis in enumerate(["X", "Y", "Z"]):
        f, ax = plt.subplots(3,1, figsize=(10, 4))
        ax = ax.ravel()
               
        ax[0].plot(frames, interp[pt][:, 0], linewidth=2, color = 'orange')
        ax[1].plot(frames, interp[pt][:, 1], linewidth=2, color = 'orange')
        ax[2].plot(frames, interp[pt][:, 2], linewidth=2, color = 'orange')
        
        ax[0].plot(frames, raw[pt][:, 0], linewidth=2, color = 'black', alpha = 0.5, label = 'x')
        ax[1].plot(frames, raw[pt][:, 1], linewidth=2, color = 'black', alpha = 0.5, label = 'y')
        ax[2].plot(frames, raw[pt][:, 2], linewidth=2, color = 'black', alpha = 0.5, label = 'z')
        
        plt.suptitle(f"{label}_({pt})_INTERPOLATED")
        plt.xlabel("Frame")
        plt.grid(True)
        for axes in ax:
            axes.legend()
        plt.tight_layout()

        out_name = f"{pt}_INTERP_v1.png"
        plt.savefig(os.path.join(fig_dir, out_name), dpi=300)
        plt.close()

print("All interpolated plots (including center) saved trial-wise.")

Plotting: Trial2_180rpmxyzpts_INTERP
Plotting: Trial2_200rpmxyzpts_INTERP
Plotting: Trial3_180rpmxyzpts_INTERP
Plotting: Trial4_400rpmxyzpts_INTERP
Plotting: Trial5_180rpmxyzpts_INTERP
Plotting: Trial5_400rpmxyzpts_INTERP
Plotting: Trial7_400rpmxyzpts_INTERP
All interpolated plots (including center) saved trial-wise.
